6강

# 드롭아웃 + 이모저모

https://youtu.be/ZDj3GRqDXwM?si=sTR5fPToBaafKMv7

- 이 minitest는 복습용으로 작은 코드를 모아둔 곳. 생각안날때 예제 보는 것과 비슷함. (혁펜하임은 이 파일을 보고 다시 상기한다고 함)

In [ ]:
import torch
from torch import nn

- w,b

In [ ]:
x=torch.randn(100,3) # 피쳐가 3개인 100개의 데이터 생성

layer=nn.Linear(3,5) # 3개의 노드 입력
print(layer(x).shape)
print(layer.weight) # 15개의 w
print(layer.bias) # 5개의 b
# 일대 w, b?

torch.Size([100, 5])
Parameter containing:
tensor([[ 0.5445, -0.0811, -0.2388],
        [-0.5648, -0.4311,  0.4065],
        [-0.5065, -0.1751,  0.5181],
        [ 0.0290,  0.0414, -0.5424],
        [-0.4683,  0.5742,  0.1989]], requires_grad=True)
Parameter containing:
tensor([ 0.2115,  0.4693,  0.4800, -0.1292, -0.2497], requires_grad=True)


- ReLU

In [ ]:
x=torch.randn(2,5)
layer=nn.ReLU() # ReLU
print(x)
print(layer(x))

tensor([[-9.4510e-01,  1.7988e+00,  9.4023e-02, -1.3274e-01,  9.1689e-01],
        [-5.9776e-01, -2.4202e+00, -1.8159e-01,  4.8431e-04, -4.5410e-01]])
tensor([[0.0000e+00, 1.7988e+00, 9.4023e-02, 0.0000e+00, 9.1689e-01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 4.8431e-04, 0.0000e+00]])


### BatchNorm (배치 정규화)

$y_i=\gamma\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}+\beta$

- BatchNorm1d

  - MLP 선형 계층의 출력값, 주로 NxC에서 (배치크기x채널(피쳐)) 사용됨.
  - **배치** **전체 데이터**의 **평균**과 **표준편차**로 구함. 노드가 3개니 평균과 표준편차도 3개임.

In [ ]:
layer=nn.BatchNorm1d(3)

# 배치 정규화 명칭이 내가 책에서 배웠던거랑 다름.
# scale(감마)이 weight, shift(베타) 이동이 bias로 됨
print(layer.weight)
print(layer.bias)

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=0))
print(layer(x).std(dim=0, unbiased=False)) # torch.std는 N-1로 나눔

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[ 1.6424,  0.4618, -1.1385],
        [ 1.5449,  0.4212, -1.0929],
        [ 0.6448, -1.5096,  1.1018],
        [-1.6199, -0.9573,  0.0715],
        [-1.0590, -0.7052, -0.6201]])
tensor([[ 1.0535,  1.1803, -0.9554],
        [ 0.9807,  1.1281, -0.9011],
        [ 0.3090, -1.3499,  1.7105],
        [-1.3809, -0.6411,  0.4845],
        [-0.9623, -0.3175, -0.3385]], grad_fn=<NativeBatchNormBackward0>)
tensor([ 0.0000e+00, -2.3842e-08, -1.1921e-08], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


- nn.LayerNorm
  - 트랜스포머나 자연어 처리 계열에서 많이 쓰임.
  - 위 처럼 배치 전체로 구하지 않고 **개별적**으로 평균과 표준편차를 구해서 정규화 시행.
  - 그렇기에 배치 크기에 영향을 받지 않음.

In [ ]:
layer=nn.LayerNorm(3)
print(layer.weight)
print(layer.bias)

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=1))
print(layer(x).std(dim=1, unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[ 1.1479,  0.8727,  0.9552],
        [-0.5213, -0.3895,  2.5269],
        [-3.0098,  0.0495, -1.4063],
        [-0.8805, -0.0109, -2.6943],
        [ 0.1461,  0.2126, -0.4124]])
tensor([[ 1.3521, -1.0334, -0.3187],
        [-0.7534, -0.6597,  1.4132],
        [-1.2440,  1.2045,  0.0394],
        [ 0.2815,  1.0595, -1.3410],
        [ 0.5850,  0.8224, -1.4075]], grad_fn=<NativeLayerNormBackward0>)
tensor([ 9.9341e-09, -3.9736e-08,  3.7253e-09, -3.9736e-08,  0.0000e+00],
       grad_fn=<MeanBackward1>)
tensor([0.9996, 1.0000, 1.0000, 1.0000, 0.9999], grad_fn=<StdBackward0>)


- nn.BatchNorm2d
  - 주로 CNN에서 많이 쓰임.
  - 입력 형태가 NxCxHxW 이럴때.
  - 1d와 비슷하게 모든 이미지들에 대한 **가로 세로 픽셀 전체**를 모아 싹 다 더한 뒤 평균과 분산을 구하고 정규화함.

In [ ]:
layer=nn.BatchNorm2d(3)
print(layer.weight)
print(layer.bias)

x = torch.randn(5,3,32,32)
print(layer(x).mean(dim=(0,2,3)))
print(layer(x).std(dim=(0,2,3), unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([ 0.0000e+00, -2.9802e-09,  1.3039e-09], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


## Dropout

- p는 죽일 확률임. p가 `0.2`면 20퍼를 끈다는겨.

- p의 확률대로 드롭아웃을 진행하고, 그만큼 **출력이 줄었**으니, 모든 노드를 켰을 때의 실제 test에서는 그만큼의 p를 곱해서 **출력을 키워**야하는게 원래 이론의 정석이지만.

  - 실제 코드에서는 test 때 속도를 높이기 위해 그냥 안그래도 드롭아웃에서 **줄어든 출력**을 **또 p로 나눠서** 더 적게만들고, 실제 test 때는 **p를 곱할 필요를 없게** 만든다.


In [ ]:
x = torch.randn(3,7)
drop = nn.Dropout(p:=0.9) # 구현에서 p는 죽일 확률 # 왈러스 연산자 사용해버리기~
print(x) # 원본 # 아래의 살아있는 데이터들과 값을 비교해라.
print(drop(x)) # 0.9면 1/10만큼 했다는 뜻. 근데 기본적으로 drop한 값들을 나누니 이 값들이 10개가 된 것!
print(drop(x)*(1-p)) # 구현은 반대로 훈련 때 1/살릴 확률을 곱하면 그대로 나옴.

drop.eval() # 실제 테스트
print(drop(x)) # 그냥 아무것도 안한 것 처럼 됨

tensor([[ 0.5215,  0.8286, -1.3542, -0.4267,  0.1634, -0.6050, -0.0929],
        [ 0.7859, -0.7337,  0.1898, -0.6065, -0.7195, -1.7617,  0.5042],
        [ 1.0270,  0.4670,  0.1434, -0.4257, -0.5421, -0.3391,  1.9614]])
tensor([[  0.0000,   0.0000, -13.5417,  -0.0000,   0.0000,  -0.0000,  -0.0000],
        [  7.8591,  -0.0000,   0.0000,  -0.0000,  -0.0000,  -0.0000,   0.0000],
        [  0.0000,   0.0000,   0.0000,  -0.0000,  -0.0000,  -0.0000,   0.0000]])
tensor([[ 0.0000,  0.0000, -0.0000, -0.4267,  0.1634, -0.0000, -0.0000],
        [ 0.7859, -0.0000,  0.0000, -0.0000, -0.0000, -0.0000,  0.0000],
        [ 1.0270,  0.0000,  0.0000, -0.4257, -0.0000, -0.0000,  0.0000]])
tensor([[ 0.5215,  0.8286, -1.3542, -0.4267,  0.1634, -0.6050, -0.0929],
        [ 0.7859, -0.7337,  0.1898, -0.6065, -0.7195, -1.7617,  0.5042],
        [ 1.0270,  0.4670,  0.1434, -0.4257, -0.5421, -0.3391,  1.9614]])


'\n...\nnn.Linear(10,100)\nnn.BatcnhNorm1d(100)\nnn.ReLU()\nnn.Dropout(p=0.5) 이런 식으로 구성!\n...\n'

In [ ]:
class sample_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop_layer=nn.Sequential(nn.Linear(5,7),
                                      nn.ReLU(),
                                      nn.Dropout(p=0.9)) # 드롭아웃 추가!

    def forward(self, x):
        x = self.drop_layer(x)
        return x

model=sample_model()
model.train() # train mode로 전환
x=torch.randn(3,5) # 그 담에 x=torch.randn(2,3,5)
print(model(x)) # 모델 안에 빈구멍이 많음.

model.eval()
print(model(x)) # test mode, 일부 0인 이유는 음수인 애들이 ReLU 때문에 사라져서

tensor([[0.0000, 4.5736, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 4.5883, 0.0000, 0.0000],
        [0.0000, 0.0000, 3.7540, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)
tensor([[0.0000, 0.4574, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9482, 0.1126, 0.0000, 1.6995, 0.4588, 1.0357, 0.0000],
        [0.0977, 0.1564, 0.3754, 0.6053, 0.2019, 0.3370, 0.6947]],
       grad_fn=<ReluBackward0>)


---